# 第3章 次の一語はサイコロで決まる：確率と温度

LLMの頭の中のサイコロを取り出して、自分で振る

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 準備：モデルを直接読み込む


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B-Instruct"
gpu = torch.cuda.is_available()
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name, dtype=torch.float16 if gpu else torch.float32, device_map="auto"
)
print("読み込み完了。置き場所：", model.device)

## LLMの「点数表」を見る


In [ ]:
text = "吾輩は猫である。名前は"
inputs = tok(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model(**inputs)

logits = out.logits[0, -1]
print(logits.shape)

In [ ]:
top = torch.topk(logits, 5)
for score, i in zip(top.values, top.indices):
    print(f"{tok.decode([int(i)])!r:8} 点数 {score.item():6.2f}")

## 点数を確率に変える：softmax


In [ ]:
import math

words = ["猫", "犬", "人間", "名前", "机"]
logits_small = [3.0, 1.6, 1.2, 0.4, -1.0]

exps = [math.exp(z) for z in logits_small]
total = sum(exps)
probs_small = [e / total for e in exps]

for w, z, e, p in zip(words, logits_small, exps, probs_small):
    print(f"{w:3}  点数 {z:5.1f}  → exp {e:6.2f}  → 確率 {p:.3f}")
print("確率の合計：", sum(probs_small))

In [ ]:
probs = torch.softmax(logits.float(), dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{tok.decode([int(i)])!r:8} 確率 {p.item()*100:5.1f}%")
print("全トークンの確率の合計：", probs.sum().item())

## サイコロを振る：サンプリング


In [ ]:
import random

def sample(words, probs):
    r = random.random()          # 0以上1未満の乱数
    total = 0.0
    for w, p in zip(words, probs):
        total += p
        if r < total:
            return w
    return words[-1]

random.seed(0)
results = [sample(words, probs_small) for _ in range(1000)]
for w in words:
    print(w, results.count(w), "回")

## 温度：LLMの「はっちゃけ度」


In [ ]:
def softmax_t(logits, T):
    exps = [math.exp(z / T) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]

for T in [0.3, 1.0, 2.0]:
    p = softmax_t(logits_small, T)
    print(f"T={T}: " + "  ".join(f"{w}{x:.2f}" for w, x in zip(words, p)))

## 自分で文章を生成する


In [ ]:
def generate(text, n_tokens=30, temperature=1.0):
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]               # ① 次のトークンの点数
        probs = torch.softmax(logits.float() / temperature, dim=-1)  # ② 温度つきで確率に
        next_id = torch.multinomial(probs, num_samples=1)   # ③ サイコロを振る
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)   # ④ 最後に付け足す
    return tok.decode(ids[0])

torch.manual_seed(42)
print(generate("吾輩は猫である。名前は", n_tokens=40, temperature=0.7))

In [ ]:
for T in [0.1, 0.7, 1.5]:
    torch.manual_seed(0)
    print(f"--- 温度 {T} ---")
    print(generate("ある日、AIが", n_tokens=40, temperature=T))

In [ ]:
inputs = tok("吾輩は猫である。名前は", return_tensors="pt").to(model.device)
torch.manual_seed(42)
out = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.7)
print(tok.decode(out[0], skip_special_tokens=True))